# 01 — Smoke test YOLOv8 (detecção de objetos em vídeo/imagem)

**Requisito do brief:** YOLOv8 customizado para detecção em vídeo clínico.

Nesta entrega (Sessão 1) demonstramos a **pipeline de detecção funcionando localmente**,
com o modelo pré-treinado `yolov8n.pt` (custo zero, sem nuvem). O *fine-tuning* para
classes clínicas específicas (instrumentos cirúrgicos ginecológicos, áreas críticas,
sangramento anômalo, objetos suspeitos) é a evolução natural e está no roadmap.

> ⚠️ **LGPD:** usar apenas imagens/vídeos sintéticos ou de domínio público. Sem PHI.

## Pré-requisito
```bash
pip install ultralytics opencv-python
```
(grupo pesado: puxa torch ~ centenas de MB; rode uma vez)

In [ ]:
# 1) Carregar o modelo pre-treinado (baixado automaticamente na 1a execucao)
from pathlib import Path
from ultralytics import YOLO

modelo = YOLO("yolov8n.pt")  # 'n' = nano: o menor e mais rapido
print("Modelo carregado. Classes conhecidas:", len(modelo.names))

In [ ]:
# 2) Obter uma imagem de exemplo (asset publico do ultralytics) e rodar a deteccao
import urllib.request

img_path = Path("../data/samples/demo_yolo.jpg")
img_path.parent.mkdir(parents=True, exist_ok=True)
if not img_path.exists():
    urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", img_path)
    print("Imagem de exemplo baixada:", img_path)

resultados = modelo(str(img_path))

In [ ]:
# 3) Listar as deteccoes (classe + confianca) e salvar a imagem anotada
r = resultados[0]
print("Objetos detectados:")
for box in r.boxes:
    cls = int(box.cls[0])
    conf = float(box.conf[0])
    print(f"  - {modelo.names[cls]}: {conf:.2f}")

out_dir = Path("../data/output")
out_dir.mkdir(parents=True, exist_ok=True)
saida = out_dir / "demo_yolo_anotada.jpg"
r.save(filename=str(saida))
print("\nImagem anotada salva em:", saida)

In [ ]:
# 4) Visualizar a imagem anotada
from IPython.display import Image
Image(filename="../data/output/demo_yolo_anotada.jpg")

## Próximos passos (customização clínica)

1. **Dataset rotulado** (sintético/domínio público) das classes-alvo escolhidas.
2. **Fine-tuning**: `YOLO('yolov8n.pt').train(data='dataset.yaml', epochs=...)`.
3. **Integração no backend**: mover a inferência para `backend/app/services/video/`
   atrás de um `VideoPort`, expondo `/api/video/analyze` e alimentando a mesma
   camada de **fusão/alertas** já usada por texto e áudio.
4. **Vídeo**: iterar quadro a quadro (`cv2.VideoCapture`) e agregar detecções por janela.